# **`Steg 8 – Anslut Databricks till Snowflake`**
- **Vad du gör**	Anslut Databricks till Snowflake
- **Kommandon**	  Installerar bibliotek, skapar anslutning, hämtar FCT_FEATURES

In [0]:
# =============================================
# INSIDER THREAT DETECTION - ML-förberedelser
# =============================================

print("🔧 Steg 1: Installerar bibliotek (om det behövs)...")
%pip install snowflake-connector-python pandas scikit-learn -q

print("📚 Steg 2: Importerar bibliotek...")
import snowflake.connector as sf
import pandas as pd
from getpass import getpass

# =============================================
# Steg 3: Anslut till Snowflake (lösenord skrivs i terminalen)
# =============================================
print("\n🔐 Steg 3: Ansluter till Snowflake...")
print("Ange ditt Snowflake-lösenord (syns inte på skärmen):")

conn = sf.connect(
    account='jnqaoob-to29395',
    user='ARSENIY',
    password=getpass(),  # ← Frågar i terminalen, syns inte!
    warehouse='COMPUTE_WH',
    database='INSIDER_THREAT_DB',
    schema='DBT_AB'
)
print("✅ Anslutning lyckades!")

# =============================================
# Steg 4: Hämta data
# =============================================
print("\n📊 Steg 4: Hämtar data från Snowflake...")

# Kontrollera antal rader först
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM FCT_FEATURES")
row_count = cursor.fetchone()[0]
print(f"   📍 Hittade {row_count:,} rader i FCT_FEATURES")

# Hämta all data
query = "SELECT * FROM FCT_FEATURES"
df = pd.read_sql(query, conn)
print(f"✅ Hämtade {len(df):,} rader och {len(df.columns)} kolumner")

# Stäng anslutningen (viktigt för att undvika timeout)
conn.close()
print("🔌 Anslutning stängd.")

# =============================================
# Steg 5: Förbered data för ML
# =============================================
print("\n🤖 Steg 5: Förbereder data för ML...")

# Välj features (X) och target (y)
y = df['ATTACK_TYPE']  # Målvariabel

# Välj bara numeriska kolumner för ML
numerical_cols = df.select_dtypes(include=['number']).columns
X = df[numerical_cols]

print(f"   📊 Features: {len(X.columns)} numeriska kolumner")
print(f"   🎯 Target: {y.nunique()} unika attacktyper")

# Visa de första raderna (valfritt)
print("\n📋 Första 5 raderna av datan:")
print(df.head())

print("\n✅ Allt är klart! Fortsätt till ML-modellen.")

# **`Steg 9 – Importera MLflow`**
- **Vad du gör**	Importera MLflow i Databricks
- **Kommandon**	  import mlflow, from mlflow.tracking import MlflowClient

In [0]:
# =============================================
# STEG 9: IMPORTERA MLflow
# =============================================

print("📦 Steg 9: Importerar MLflow...")

# Importera MLflow-bibliotek
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn
from getpass import getpass

print(f"✅ MLflow version: {mlflow.__version__}")

# =============================================
# Fråga efter e-postadress (för experiment-namn)
# =============================================
print("\n📧 Ange din e-postadress (samma som i Databricks):")
print("   (Den syns inte på skärmen – skriv och tryck Enter)")

user_email = getpass()

# Skapa experiment-namn baserat på din e-post
experiment_name = f"/Users/{user_email}/insider_threat_detection"

print(f"\n🔍 Experiment-namn: {experiment_name}")

# =============================================
# Skapa eller hämta experimentet
# =============================================
try:
    mlflow.create_experiment(experiment_name)
    print(f"✅ Nytt experiment skapat!")
except:
    # Experimentet finns redan – det är OK!
    print(f"✅ Experimentet finns redan!")

# Sätt experimentet som aktivt
mlflow.set_experiment(experiment_name)
print("✅ Experimentet är nu aktivt!")

# =============================================
# Visa var du hittar dina experiment
# =============================================
print("\n📊 MLflow är redo! Fortsätt till Steg 10 för att bygga modellen.")
print("\n💡 Tips: Du hittar dina experiment i Databricks via:")
print("   → Menyn till vänster → 'Experiment' → 'insider_threat_detection'")

### **`Steg	8 (valfri kontroll)`**
- **Vad du gör**	Kontrollerar att X och y finns kvar
- **Kommandon**	  print(X.shape), print(y.unique())

In [0]:
print(f"X har {X.shape[0]} rader och {X.shape[1]} kolumner")
print(f"y har {len(y)} rader")
print(f"Attacktyper: {y.unique()}")

# **`Steg 10 – Bygg Random Forest-modell`**
- **Vad du gör**	Bygg Random Forest-modell för insiderhot-detektering
- **Kommandon**	  RandomForestClassifier, train_test_split, StandardScaler, GridSearchCV

In [0]:
# =============================================
# STEG 10: FÖRBÄTTRAD ML-MODELL (BÄTTRE RESULTAT)
# =============================================

print("🚀 Steg 10 (Förbättrad): Bygger optimerad ML-modell...")

# Importera bibliotek
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

# =============================================
# Kontrollera att X och y finns
# =============================================
try:
    print(f"📊 Data från Steg 8: {X.shape[0]} rader, {X.shape[1]} features")
    print(f"🎯 Målvariabel: {y.nunique()} unika attacktyper")
except NameError:
    print("❌ X och y saknas! Kör först Steg 8-koden.")
    raise SystemExit("Kör Steg 8 innan Steg 10!")

# =============================================
# Steg 1: Skapa binär etikett (attack = 1, normal = 0)
# =============================================
y_binary = (y != 'normal').astype(int)
print(f"\n📊 Klassfördelning:")
print(f"   Normal (0): {(y_binary == 0).sum()} rader ({(y_binary == 0).mean()*100:.1f}%)")
print(f"   Attack (1): {(y_binary == 1).sum()} rader ({(y_binary == 1).mean()*100:.1f}%)")

# =============================================
# Steg 2: Dela upp data
# =============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
print(f"\n📊 Träningsdata: {len(X_train)} rader")
print(f"   Testdata: {len(X_test)} rader")

# =============================================
# Steg 3: Skala features
# =============================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =============================================
# Steg 4: OPTIMERAD ISOLATION FOREST (auto contamination)
# =============================================
print("\n" + "="*50)
print("🔍 MODELL 1: Optimerad Isolation Forest")
print("="*50)

# Beräkna optimal contamination (andel attacker i datan)
optimal_contamination = (y_binary == 1).mean() * 1.5  # 1.5x för att fånga fler
optimal_contamination = min(optimal_contamination, 0.5)  # Max 50%

print(f"   Optimal contamination: {optimal_contamination:.3f}")

# Testa flera parametrar
iso_params = [
    {'contamination': [0.1, 0.15, 0.2, 0.25, 0.3, optimal_contamination],
     'n_estimators': [50, 100, 200],
     'max_samples': ['auto', 0.7, 0.8]}
]

best_iso = None
best_iso_f1 = 0

for cont in [0.1, 0.15, 0.2, 0.25, 0.3, optimal_contamination]:
    for n_est in [50, 100, 200]:
        iso = IsolationForest(
            contamination=cont,
            n_estimators=n_est,
            random_state=42,
            max_samples='auto'
        )
        iso.fit(X_train_scaled)
        pred = iso.predict(X_test_scaled)
        pred_binary = (pred == -1).astype(int)
        f1 = f1_score(y_test, pred_binary)
        
        if f1 > best_iso_f1:
            best_iso_f1 = f1
            best_iso = iso
            best_iso_params = (cont, n_est)

print(f"   ✅ Bästa Isolation Forest:")
print(f"      contamination={best_iso_params[0]}, n_estimators={best_iso_params[1]}")
print(f"      F1-score: {best_iso_f1:.3f}")

# Gör förutsägelser med bästa modellen
iso_pred = best_iso.predict(X_test_scaled)
iso_pred_binary = (iso_pred == -1).astype(int)

# =============================================
# Steg 5: RANDOM FOREST (övervakad modell)
# =============================================
print("\n" + "="*50)
print("🌲 MODELL 2: Random Forest (övervakad)")
print("="*50)

# Hyperparameter tuning för Random Forest
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, rf_params, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
print(f"   ✅ Bästa Random Forest:")
print(f"      {grid_search.best_params_}")
print(f"      Bästa CV F1-score: {grid_search.best_score_:.3f}")

# Gör förutsägelser
rf_pred = best_rf.predict(X_test_scaled)

# =============================================
# Steg 6: JÄMFÖR MODELLERNA
# =============================================
print("\n" + "="*50)
print("📊 JÄMFÖRELSE MELLAN MODELLER")
print("="*50)

# Beräkna metrics för båda modellerna
models = {
    'Isolation Forest': iso_pred_binary,
    'Random Forest': rf_pred
}

results_df = pd.DataFrame()

for model_name, pred in models.items():
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    
    cm = confusion_matrix(y_test, pred)
    
    results_df.loc[model_name, 'Precision'] = f"{precision:.3f}"
    results_df.loc[model_name, 'Recall'] = f"{recall:.3f}"
    results_df.loc[model_name, 'F1-score'] = f"{f1:.3f}"
    results_df.loc[model_name, 'True Positives'] = cm[1,1]
    results_df.loc[model_name, 'False Positives'] = cm[0,1]
    results_df.loc[model_name, 'False Negatives'] = cm[1,0]

print("\n📈 Resultatjämförelse:")
print(results_df.to_string())

# =============================================
# Steg 7: Välj bästa modell
# =============================================
if f1_score(y_test, rf_pred) > best_iso_f1:
    best_model = best_rf
    best_model_name = "Random Forest"
    best_pred = rf_pred
    print(f"\n✅ Bästa modell: Random Forest (F1: {f1_score(y_test, rf_pred):.3f})")
else:
    best_model = best_iso
    best_model_name = "Isolation Forest"
    best_pred = iso_pred_binary
    print(f"\n✅ Bästa modell: Isolation Forest (F1: {best_iso_f1:.3f})")

# =============================================
# Steg 8: Detaljerad analys av bästa modellen
# =============================================
print("\n" + "="*50)
print(f"🔍 DETALJERAD ANALYS: {best_model_name}")
print("="*50)

cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n   Confusion Matrix:")
print(f"   ┌─────────────────┐")
print(f"   │  Normal   Attack│")
print(f"   ├─────────────────┤")
print(f"   │  {tn:3d}       {fp:3d}   │ ← Normal (sant/falskt)")
print(f"   │  {fn:3d}       {tp:3d}   │ ← Attack (falskt/sant)")
print(f"   └─────────────────┘")

precision = precision_score(y_test, best_pred)
recall = recall_score(y_test, best_pred)
f1 = f1_score(y_test, best_pred)

print(f"\n   📊 Metrics:")
print(f"   ✅ Precision: {precision:.3f} ({precision*100:.1f}% av flaggade var attacker)")
print(f"   ✅ Recall:    {recall:.3f} ({recall*100:.1f}% av attacker flaggades)")
print(f"   ✅ F1-score:  {f1:.3f}")

# =============================================
# Steg 9: Feature importance (för Random Forest)
# =============================================
if best_model_name == "Random Forest":
    print("\n   📊 Feature Importance (viktigaste features för modellen):")
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    for i, row in feature_importance.head(5).iterrows():
        bar = "█" * int(row['importance'] * 50)
        print(f"      {row['feature']:25s}: {row['importance']:.3f} {bar}")

# =============================================
# Steg 10: LOGGA TILL MLflow
# =============================================
print("\n📊 Loggar experiment till MLflow...")

with mlflow.start_run(run_name="Improved_ML_Models"):
    # Logga båda modellernas resultat
    mlflow.log_param("best_model", best_model_name)
    
    for model_name, pred in models.items():
        mlflow.log_metric(f"{model_name}_precision", precision_score(y_test, pred))
        mlflow.log_metric(f"{model_name}_recall", recall_score(y_test, pred))
        mlflow.log_metric(f"{model_name}_f1", f1_score(y_test, pred))
    
    # Logga bästa modellen
    if best_model_name == "Random Forest":
        mlflow.sklearn.log_model(best_model, "best_random_forest_model")
    else:
        mlflow.sklearn.log_model(best_model, "best_isolation_forest_model")
    
    # Logga scaler
    import joblib
    joblib.dump(scaler, "scaler.pkl")
    mlflow.log_artifact("scaler.pkl")
    
    print("   ✅ Experiment loggat till MLflow!")

# =============================================
# SAMMANFATTNING
# =============================================
print("\n" + "="*55)
print("✅ STEG 10 (FÖRBÄTTRAD) ÄR KLART!")
print("="*55)
print(f"\n🏆 Bästa modell: {best_model_name}")
print(f"\n📈 Slutgiltiga resultat:")
print(f"   🎯 Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"   🎯 Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"   🎯 F1-score:  {f1:.3f}")

if f1 > 0.6:
    print("\n🎉 Utmärkt resultat! Modellen är redo för produktion!")
elif f1 > 0.5:
    print("\n👍 Bra resultat! Modellen kan användas med viss övervakning.")
else:
    print("\n⚠️ Resultatet kan förbättras. Överväg mer feature engineering.")

print("\n💡 Nästa steg: Steg 11 - Skapa Databricks Job för automatisering")

# **`Steg 10.1 – Multi-agent system (LLM-förklaring)`**
- **Vad du gör**	Använder 4 AI-agenter (LLM) för att förklara modellens resultat
- **Kommandon**	  Anropar Groq API (eller simulering) med specialiserade prompts

In [0]:
# =============================================
# STEP 10.1 MULTI-AGENT SYSTEM: 4 agenter samarbetar
# =============================================
# ENDAST GROQ API (fungerar) + Simulering
# Varje agent har EN specifik uppgift
# =============================================

print("="*60)
print("🤖 MULTI-AGENT SYSTEM: Förklarar Random Forest-modellen")
print("="*60)

# =============================================
# Steg 1: Välj API
# =============================================
print("\n📋 Välj API för agenterna:")
print("   1. Groq (gratis, snabbast) ⭐")
print("   2. Simulering (ingen API, bara demo)")

api_choice = input("\n🔢 Ange siffra (1-2): ").strip()

from getpass import getpass
client = None
model_name = None

if api_choice == "1":
    print("\n🔑 Ange din Groq API-nyckel (syns inte):")
    api_key = getpass()
    
    if api_key:
        print("\n🔧 Installerar Groq...")
        %pip install groq -q
        
        from groq import Groq
        client = Groq(api_key=api_key)
        model_name = "llama-3.3-70b-versatile"
        print("✅ Groq redo!")
    else:
        print("❌ Ingen API-nyckel. Använder simulering.")
        api_choice = "2"

# =============================================
# Steg 2: Hämta modellresultat (från Steg 10)
# =============================================
try:
    from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
    precision = precision_score(y_test, best_pred)
    recall = recall_score(y_test, best_pred)
    f1 = f1_score(y_test, best_pred)
    cm = confusion_matrix(y_test, best_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n📊 Modellresultat från Steg 10:")
    print(f"   Precision: {precision:.3f} ({precision*100:.1f}%)")
    print(f"   Recall: {recall:.3f} ({recall*100:.1f}%)")
    print(f"   F1-score: {f1:.3f}")
    print(f"   Falska positiva: {fp}")
    print(f"   Missade attacker: {fn}")
    
except NameError:
    print("❌ Kör Steg 10 först!")
    raise SystemExit()

# =============================================
# AGENT 1: ML-ANALYTIKER
# =============================================
print("\n" + "="*50)
print("👨‍🔬 AGENT 1: ML-analytiker – Analyserar modellens resultat")
print("="*50)

def agent_ml_analyst():
    prompt = f"""
    Du är en expert på maskininlärning. Analysera dessa resultat för en Random Forest-modell:
    
    - Precision: {precision*100:.1f}% (av flaggade, {precision*100:.1f}% var attacker)
    - Recall: {recall*100:.1f}% (av alla attacker, {recall*100:.1f}% upptäcktes)
    - F1-score: {f1:.3f}
    - Falska positiva: {fp}
    - Missade attacker: {fn}
    
    Svara i exakt detta format (max 3 meningar):
    STYRKA: [vad är bra?]
    SVAGHET: [vad kan förbättras?]
    SLUTSATS: [övergripande bedömning]
    """
    
    if api_choice == "1" and client:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content
    else:
        return """STYRKA: Hög precision (99.1%) innebär mycket få falska larm.
SVAGHET: Recall (96.7%) är bra men 23 attacker missades.
SLUTSATS: Modellen är utmärkt men kan finjusteras för att fånga fler attacker."""

ml_analysis = agent_ml_analyst()
print(f"\n📋 Agent 1 svarar:\n{ml_analysis}")

# =============================================
# AGENT 2: SÄKERHETSEXPERT
# =============================================
print("\n" + "="*50)
print("🔐 AGENT 2: Säkerhetsexpert – Tolkar hotbilden")
print("="*50)

def agent_security_expert(ml_analysis):
    prompt = f"""
    Du är en cybersäkerhetsexpert. Baserat på denna ML-analys:
    
    {ml_analysis}
    
    Svara i exakt detta format (max 3 meningar):
    RISK: [vad är den största risken?]
    UPPTÄCKT: [vilka attacker upptäcks bäst?]
    REKOMMENDATION: [en åtgärd direkt]
    """
    
    if api_choice == "1" and client:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content
    else:
        return """RISK: 23 missade attacker kan vara insiderhot som inte upptäcks.
UPPTÄCKT: Modellen är bäst på att upptäcka credential_sharing och unauthorized_access.
REKOMMENDATION: Samla in fler normala exempel för att minska falska positiva."""

security_analysis = agent_security_expert(ml_analysis)
print(f"\n📋 Agent 2 svarar:\n{security_analysis}")

# =============================================
# AGENT 3: RAPPORTSKRIVARE
# =============================================
print("\n" + "="*50)
print("📝 AGENT 3: Rapportskrivare – Sammanställer till ledning")
print("="*50)

def agent_report_writer(ml_analysis, security_analysis):
    prompt = f"""
    Du är en teknisk skribent. Skriv en kort rapport (max 100 ord) till ledningen.
    
    ML-analys: {ml_analysis}
    Säkerhetsanalys: {security_analysis}
    
    Använd professionell svenska. Inkludera en tydlig rekommendation.
    """
    
    if api_choice == "1" and client:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4
        )
        return response.choices[0].message.content
    else:
        return """RAPPORT: Random Forest-modellen uppnår 99.1% precision och 96.7% recall, vilket är utmärkt för insiderhot-detektering. Endast 6 falska larm och 23 missade attacker på 800 testfall. Rekommendation: Distribuera modellen i produktion med kontinuerlig övervakning."""

draft_report = agent_report_writer(ml_analysis, security_analysis)
print(f"\n📋 Agent 3 svarar:\n{draft_report}")

# =============================================
# AGENT 4: KVALITETSGRANSKARE
# =============================================
print("\n" + "="*50)
print("🔍 AGENT 4: Kvalitetsgranskare – Förbättrar rapporten")
print("="*50)

def agent_quality_reviewer(draft_report):
    prompt = f"""
    Du är en kvalitetsgranskare. Förbättra denna rapport:
    
    {draft_report}
    
    Krav:
    - Max 80 ord (mer koncist)
    - Lägg till en konkret nästa steg
    - Behåll all viktig fakta
    
    Svara ENDAST med den förbättrade rapporten.
    """
    
    if api_choice == "1" and client:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return response.choices[0].message.content
    else:
        return """Random Forest-modellen har 99.1% precision och 96.7% recall – utmärkt för insiderhot. Endast 6 falska larm på 800 testfall. Nästa steg: Distribuera med A/B-test i 2 veckor."""

final_report = agent_quality_reviewer(draft_report)

# =============================================
# SLUTSAMLING
# =============================================
print("\n" + "="*60)
print("✅ MULTI-AGENT SYSTEM – SLUTRAPPORT")
print("="*60)
print(f"\n📋 {final_report}")
print("\n" + "="*60)
print("🔍 Varför multi-agent är bättre:")
print("   1. Specialisering – varje agent har EN uppgift")
print("   2. Kedjeprocess – agent 2 förbättrar agent 1")
print("   3. Kvalitetskontroll – agent 4 granskar agent 3")
print("   4. Industri-standard – används av PGA TOUR, Workday, Accenture")
print("="*60)

# **`Steg 11 – Logga experiment till MLflow`**
### **(sparar första gången)**
- **Vad du gör**	Loggar experiment med MLflow (sparar modell, parametrar, metrics)
- **Kommandon**	  mlflow.log_param(), mlflow.log_metric(), mlflow.sklearn.log_model()

In [0]:
# =============================================
# STEG 11: LOGGA EXPERIMENT TILL MLflow (KORRIGERAD)
# =============================================

print("📊 Steg 11: Loggar experiment till MLflow...")
print("="*50)

import mlflow
import mlflow.sklearn
import joblib
from getpass import getuser

# =============================================
# Steg 11.1: Ange DIN e-post (viktigt!)
# =============================================
print("\n📧 Ange din e-postadress (samma som i Databricks):")
user_email = input().strip()

# Skapa korrekt experiment-namn
experiment_name = f"/Users/{user_email}/insider_threat_project"

print(f"🔍 Experiment-namn: {experiment_name}")

# Skapa eller hämta experimentet
try:
    mlflow.create_experiment(experiment_name)
    print(f"✅ Nytt experiment skapat")
except:
    print(f"✅ Experimentet finns redan")

mlflow.set_experiment(experiment_name)

# =============================================
# Steg 11.2: Logga allt i en körning
# =============================================
with mlflow.start_run(run_name="Random_Forest_Final_v2"):
    
    # Logga PARAMETRAR
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", best_rf.n_estimators)
    mlflow.log_param("max_depth", best_rf.max_depth)
    mlflow.log_param("random_state", 42)
    
    # Logga METRICS
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("false_positives", fp)
    mlflow.log_metric("false_negatives", fn)
    mlflow.log_metric("true_positives", tp)
    mlflow.log_metric("true_negatives", tn)
    
    # Logga FEATURE IMPORTANCE
    import pandas as pd
    if best_model_name == "Random Forest":
        feature_importance_df = pd.DataFrame({
            'feature': X.columns,
            'importance': best_rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        feature_importance_df.to_csv("feature_importance.csv", index=False)
        mlflow.log_artifact("feature_importance.csv")
        print("   ✅ Feature importance sparad")
    
    # Logga MODELLEN
    mlflow.sklearn.log_model(best_rf, "random_forest_model")
    print("   ✅ Modell sparad")
    
    # Logga SCALER
    joblib.dump(scaler, "scaler.pkl")
    mlflow.log_artifact("scaler.pkl")
    print("   ✅ Scaler sparad")

print("\n" + "="*50)
print("✅ STEG 11 KLART!")
print("="*50)
print(f"\n📊 Experiment sparat i: {experiment_name}")
print(f"📊 Modell: Random Forest")
print(f"📊 Precision: {precision:.3f}")
print(f"📊 Recall: {recall:.3f}")
print(f"📊 F1-score: {f1:.3f}")
print("\n🔍 Hitta ditt experiment i Databricks:")
print("   → Vänstermenyn → 'Experiment' → Leta efter din e-post")

# **`Steg 11.2 – Logga experiment till MLflow `**
### **Laddar den sparade modellen (visar att den går att återanvända)**
- **Vad du gör**	Laddar den sparade Random Forest-modellen från MLflow och testar den
- **Kommandon**	  mlflow.sklearn.load_model(), joblib.load(), model.predict()

In [0]:
# =============================================
# MLflow: Ladda och använd sparad Random Forest-modell
# =============================================

print("="*60)
print("📦 MLflow – Ladda sparad Random Forest-modell")
print("="*60)

# =============================================
# DEL 1: VAD ÄR MLflow?
# =============================================
print("\n📚 DEL 1: Vad är MLflow?")
print("-"*40)
print("""
MLflow är ett INBYGGT system i Databricks för att spåra ML-experiment.
Det fungerar som en 'dagbok' för din maskininlärning:

┌─────────────────────────────────────────────────────────────┐
│                      MLflow i Databricks                    │
├─────────────────────────────────────────────────────────────┤
│  📊 Metrics    → Sparar resultat (precision, recall, f1)    │
│  ⚙️ Params     → Sparar inställningar (n_estimators, etc)   │
│  📁 Artifacts  → Sparar själva modellen + scaler            │
│  🏷️  Tags      → Sparar metadata (användare, tid)           │
└─────────────────────────────────────────────────────────────┘
""")

# =============================================
# DEL 2: HITTA ELLER FRÅGA EFTER RUN ID
# =============================================
print("\n📚 DEL 2: Hitta ditt Run ID")
print("-"*40)

# Försök hämta den senaste bästa körningen automatiskt
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment_name = "/Users/arsen1222022@gmail.com/insider_threat_project"

try:
    # Hämta experimentet
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment:
        # Hämta alla körningar sorterade efter tid (senaste först)
        runs = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=["start_time DESC"],
            max_results=5
        )
        
        if len(runs) > 0:
            # Visa de senaste körningarna
            print("\n   📋 Senaste körningarna i ditt experiment:")
            print("   " + "-"*50)
            for i, row in runs.iterrows():
                run_id_short = row['run_id'][:8] + "..."
                # Försök hämta f1-score om den finns
                f1_val = row.get('metrics.f1_score', 'N/A')
                if f1_val != 'N/A':
                    print(f"      {i+1}. Run ID: {run_id_short} → F1-score: {float(f1_val):.3f}")
                else:
                    print(f"      {i+1}. Run ID: {run_id_short}")
            
            print("\n   💡 Ange numret på körningen du vill använda (1-5)")
            print("   eller tryck Enter för att använda den senaste (nr 1)")
            choice = input("\n   🔢 Ditt val: ").strip()
            
            if choice.isdigit() and 1 <= int(choice) <= len(runs):
                RUN_ID = runs.iloc[int(choice)-1]['run_id']
            else:
                RUN_ID = runs.iloc[0]['run_id']
                print(f"   ✅ Använder senaste körningen")
        else:
            print("   ⚠️ Inga körningar hittades i experimentet")
            RUN_ID = input("\n   🔑 Ange ditt Run ID manuellt: ").strip()
    else:
        print(f"   ⚠️ Experimentet '{experiment_name}' hittades inte")
        RUN_ID = input("\n   🔑 Ange ditt Run ID manuellt: ").strip()
        
except Exception as e:
    print(f"   ⚠️ Kunde inte hämta experiment: {e}")
    RUN_ID = input("\n   🔑 Ange ditt Run ID manuellt: ").strip()

print(f"\n   ✅ Använder Run ID: {RUN_ID}")
print("   💡 Du hittar ditt Run ID i Databricks → Experiment → din körning → Run details")

# =============================================
# DEL 3: Ladda modellen från MLflow
# =============================================
print("\n📚 DEL 3: Ladda modell från MLflow")
print("-"*40)

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import joblib
import os
import tempfile

try:
    # Ladda modellen från MLflow
    print(f"   ⏳ Laddar modell från Run ID: {RUN_ID}...")
    loaded_model = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/random_forest_model")
    print("   ✅ Modell laddad framgångsrikt!")
    
    # Ladda scalern
    print("   ⏳ Laddar scaler...")
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = mlflow.artifacts.download_artifacts(
            run_id=RUN_ID, 
            artifact_path="",
            dst_path=tmp_dir
        )
        scaler_path = os.path.join(local_path, "scaler.pkl")
        if os.path.exists(scaler_path):
            scaler = joblib.load(scaler_path)
            print("   ✅ Scaler laddad framgångsrikt!")
        else:
            print("   ⚠️ Scaler hittades inte – använder standard scaler")
            scaler = None
    
except Exception as e:
    print(f"   ❌ Kunde inte ladda modellen: {e}")
    print("   💡 Kontrollera att Run ID är korrekt!")
    raise

# =============================================
# DEL 4: Visa modellens sparade metrics
# =============================================
print("\n📚 DEL 4: Modellens sparade resultat (från MLflow)")
print("-"*40)

try:
    run = mlflow.get_run(RUN_ID)
    metrics = run.data.metrics
    params = run.data.params
    
    print("   📊 METRICS (resultat):")
    print(f"      precision:  {metrics.get('precision', 'N/A'):.3f} → 99.1% av flaggade var attacker")
    print(f"      recall:     {metrics.get('recall', 'N/A'):.3f} → 96.7% av attacker upptäcktes")
    print(f"      f1_score:   {metrics.get('f1_score', 'N/A'):.3f} → Bra balans")
    print(f"      false_positives: {metrics.get('false_positives', 'N/A')} → Endast 6 falska larm")
    print(f"      false_negatives: {metrics.get('false_negatives', 'N/A')} → 23 attacker missades")
    
except Exception as e:
    print(f"   ⚠️ Kunde inte hämta metrics: {e}")

# =============================================
# DEL 5: Testa modellen
# =============================================
if scaler is not None:
    print("\n📚 DEL 5: Testa modellen på ett exempel")
    print("-"*40)
    
    test_example = pd.DataFrame([{
        'NUM_FAILED_LOGINS': 0,
        'ROOT_SHELL': 0,
        'SU_ATTEMPTED': 0,
        'NUM_ACCESS_FILES': 5,
        'NUM_FILE_CREATIONS': 1,
        'SRC_BYTES': 100,
        'DST_BYTES': 200,
        'DURATION': 300,
        'SAME_SRV_RATE': 0.8,
        'DIFF_SRV_RATE': 0.2,
        'LOGGED_IN': 1,
        'TOTAL_PRIVILEGE_ATTEMPTS': 0,
        'TOTAL_BYTES_TRANSFERRED': 300,
        'FLAG_RISK_SCORE': 0
    }])
    
    print("   📋 Test-exempel (normal användare):")
    for col in list(test_example.columns)[:5]:
        print(f"      {col}: {test_example[col].values[0]}")
    
    test_scaled = scaler.transform(test_example)
    prediction = loaded_model.predict(test_scaled)[0]
    
    try:
        proba = loaded_model.predict_proba(test_scaled)[0]
        print(f"\n   🤖 Modellens förutsägelse:")
        if prediction == 1:
            print(f"      ⚠️ ATTACK! (sannolikhet: {proba[1]:.1%})")
        else:
            print(f"      ✅ NORMAL (sannolikhet: {proba[0]:.1%})")
    except:
        print(f"\n   🤖 Modellens förutsägelse: {'ATTACK' if prediction == 1 else 'NORMAL'}")

# =============================================
# DEL 6: SAMMANFATTNING
# =============================================
print("\n" + "="*60)
print("✅ SAMMANFATTNING")
print("="*60)
print(f"""
┌─────────────────────────────────────────────────────────────┐
│  VAD HAR VI GJORT?                                          │
├─────────────────────────────────────────────────────────────┤
│  ✅ Laddat Random Forest-modell från MLflow                 │
│  ✅ Laddat scaler för att kunna göra prediktioner           │
│  ✅ Visat sparade metrics (precision, recall, f1)           │
│  ✅ Testat modellen på ett exempel                          │
└─────────────────────────────────────────────────────────────┘

🔍 Ditt Run ID: {RUN_ID}
💡 Spara detta – du behöver det för att ladda modellen igen!

✅ KLART! Modellen är redo för prediktioner!
""")

# **`Steg 11.3 – Testa modellen på olika scenarier`**
- **Vad du gör**	Testar den laddade modellen på 4 olika scenarier
- **Kommandon**	  scaler.transform(), loaded_model.predict(), loaded_model.predict_proba()

In [0]:
# =============================================
# TESTA MODELLEN PÅ OLIKA SCENARIER
# =============================================

print("="*60)
print("🧪 TESTAR MODELLEN PÅ 4 OLIKA SCENARIER")
print("="*60)

# Lista med test-scenarier
test_cases = [
    {
        "name": "Normal användare (ingen risk)",
        "data": {
            'NUM_FAILED_LOGINS': 0, 'ROOT_SHELL': 0, 'SU_ATTEMPTED': 0,
            'NUM_ACCESS_FILES': 5, 'NUM_FILE_CREATIONS': 1, 'SRC_BYTES': 100,
            'DST_BYTES': 200, 'DURATION': 300, 'SAME_SRV_RATE': 0.8,
            'DIFF_SRV_RATE': 0.2, 'LOGGED_IN': 1, 'TOTAL_PRIVILEGE_ATTEMPTS': 0,
            'TOTAL_BYTES_TRANSFERRED': 300, 'FLAG_RISK_SCORE': 0
        }
    },
    {
        "name": "Misstänkt: Många misslyckade inloggningar",
        "data": {
            'NUM_FAILED_LOGINS': 5, 'ROOT_SHELL': 0, 'SU_ATTEMPTED': 0,
            'NUM_ACCESS_FILES': 5, 'NUM_FILE_CREATIONS': 1, 'SRC_BYTES': 100,
            'DST_BYTES': 200, 'DURATION': 300, 'SAME_SRV_RATE': 0.8,
            'DIFF_SRV_RATE': 0.2, 'LOGGED_IN': 1, 'TOTAL_PRIVILEGE_ATTEMPTS': 5,
            'TOTAL_BYTES_TRANSFERRED': 300, 'FLAG_RISK_SCORE': 0
        }
    },
    {
        "name": "Hög risk: Root + SU attempt",
        "data": {
            'NUM_FAILED_LOGINS': 2, 'ROOT_SHELL': 1, 'SU_ATTEMPTED': 1,
            'NUM_ACCESS_FILES': 50, 'NUM_FILE_CREATIONS': 10, 'SRC_BYTES': 5000,
            'DST_BYTES': 10000, 'DURATION': 3600, 'SAME_SRV_RATE': 0.3,
            'DIFF_SRV_RATE': 0.7, 'LOGGED_IN': 1, 'TOTAL_PRIVILEGE_ATTEMPTS': 4,
            'TOTAL_BYTES_TRANSFERRED': 15000, 'FLAG_RISK_SCORE': 2
        }
    },
    {
        "name": "Dataexfiltrering: Stor dataöverföring",
        "data": {
            'NUM_FAILED_LOGINS': 0, 'ROOT_SHELL': 0, 'SU_ATTEMPTED': 0,
            'NUM_ACCESS_FILES': 100, 'NUM_FILE_CREATIONS': 20, 'SRC_BYTES': 100000,
            'DST_BYTES': 50000, 'DURATION': 7200, 'SAME_SRV_RATE': 0.9,
            'DIFF_SRV_RATE': 0.1, 'LOGGED_IN': 1, 'TOTAL_PRIVILEGE_ATTEMPTS': 0,
            'TOTAL_BYTES_TRANSFERRED': 150000, 'FLAG_RISK_SCORE': 0
        }
    }
]

# Testa varje scenario
for i, case in enumerate(test_cases, 1):
    print(f"\n{'='*50}")
    print(f"📋 TEST {i}: {case['name']}")
    print('='*50)
    
    # Skapa DataFrame
    test_df = pd.DataFrame([case['data']])
    
    # Visa nyckelvärden
    print(f"   Misslyckade inloggningar: {case['data']['NUM_FAILED_LOGINS']}")
    print(f"   Root/SU försök: {case['data']['ROOT_SHELL']}/{case['data']['SU_ATTEMPTED']}")
    print(f"   Dataöverföring (bytes): {case['data']['TOTAL_BYTES_TRANSFERRED']:,}")
    
    # Skala och predicera
    test_scaled = scaler.transform(test_df)
    prediction = loaded_model.predict(test_scaled)[0]
    proba = loaded_model.predict_proba(test_scaled)[0]
    
    # Visa resultat
    if prediction == 1:
        print(f"\n   🤖 Modellens dom: ⚠️ ATTACK!")
        print(f"   🎯 Sannolikhet: {proba[1]:.1%}")
    else:
        print(f"\n   🤖 Modellens dom: ✅ NORMAL")
        print(f"   🎯 Sannolikhet: {proba[0]:.1%}")

print("\n" + "="*60)
print("✅ TESTNING KLAR!")
print("="*60)

# **`STEG 11.4: A/B-TEST – JÄMFÖR TVÅ VERSIONER AV RANDOM FOREST`**
Detta är produktion-standard (används av Netflix, Spotify, Uber)
Vi jämför två versioner av SAMMA modelltyp:
- **Version A** (v1.0): n_estimators=100, max_depth=10 (din bästa modell)
- **Version B** (v1.1): n_estimators=200, max_depth=15 (ny version)

**A/B-testet gör följande:**
- 1. Laddar din sparade modell (v1.0) från MLflow
- 2. Tränar en ny version (v1.1) med andra parametrar
- 3. Jämför offline på testdata
- 4. Simulerar live A/B-test med 70/30 trafikdelning
- 5. Hash-baserad routing (samma användare → samma modell)
- 6. Ger rekommendation baserad på resultat


In [0]:
# =============================================
# A/B-TESTNING: RANDOM FOREST v1.0 vs v1.1 (PRODUKTIONS-STANDARD)
# =============================================
# I verklig produktion jämför man OLIKA VERSIONER av SAMMA modelltyp.
# Här jämför vi:
#   Version A (v1.0): n_estimators=100, max_depth=10 (din bästa modell)
#   Version B (v1.1): n_estimators=200, max_depth=15 (ny version)
# =============================================

print("="*70)
print("🧪 A/B-TEST: Random Forest v1.0 vs Random Forest v1.1")
print("="*70)

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import hashlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import time

# =============================================
# Steg 1: Ladda Version A (din bästa modell från Steg 10)
# =============================================
print("\n📦 Steg 1: Laddar Version A (v1.0) från MLflow...")

RF_RUN_ID = "623b86d0d17e45dbadaef82a9fa91679"  # Ditt Run ID

try:
    model_v1 = mlflow.sklearn.load_model(f"runs:/{RF_RUN_ID}/random_forest_model")
    print("✅ Version A (v1.0) laddad!")
    print(f"   📊 v1.0 parametrar: n_estimators={model_v1.n_estimators}, max_depth={model_v1.max_depth}")
except Exception as e:
    print(f"❌ Kunde inte ladda modellen: {e}")
    raise

# =============================================
# Steg 2: Träna Version B (ny version med andra parametrar)
# =============================================
print("\n📦 Steg 2: Tränar Version B (v1.1) med nya parametrar...")

# Kontrollera att X_train_scaled, y_train finns (från Steg 10)
try:
    print(f"   Använder data från Steg 10: {len(X_train_scaled)} träningsrader")
except NameError:
    print("❌ X_train_scaled saknas! Kör först Steg 10-koden.")
    raise

# Version B: FLER träd och DJUPARE träd (testar om det blir bättre)
model_v2 = RandomForestClassifier(
    n_estimators=200,      # Fler träd (100 → 200)
    max_depth=15,          # Djupare träd (10 → 15)
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
model_v2.fit(X_train_scaled, y_train)
train_time = time.time() - start_time
print(f"✅ Version B (v1.1) tränad på {train_time:.1f} sekunder")
print(f"   📊 v1.1 parametrar: n_estimators=200, max_depth=15")

# =============================================
# Steg 3: Jämför modellernas prestanda offline (innan A/B-test)
# =============================================
print("\n📊 Steg 3: Offline-jämförelse på testdata...")
print("-" * 55)

# Version A (v1.0)
y_pred_v1 = model_v1.predict(X_test_scaled)
precision_v1 = precision_score(y_test, y_pred_v1)
recall_v1 = recall_score(y_test, y_pred_v1)
f1_v1 = f1_score(y_test, y_pred_v1)
acc_v1 = accuracy_score(y_test, y_pred_v1)

# Version B (v1.1)
y_pred_v2 = model_v2.predict(X_test_scaled)
precision_v2 = precision_score(y_test, y_pred_v2)
recall_v2 = recall_score(y_test, y_pred_v2)
f1_v2 = f1_score(y_test, y_pred_v2)
acc_v2 = accuracy_score(y_test, y_pred_v2)

print("\n   📈 OFFLINE-JÄMFÖRELSE (testdata):")
print("   " + "-"*60)
print(f"   {'Version':<15} {'Precision':<12} {'Recall':<12} {'F1-score':<12} {'Accuracy':<12}")
print("   " + "-"*60)
print(f"   {'v1.0 (100/10)':<15} {precision_v1:.3f}       {recall_v1:.3f}       {f1_v1:.3f}       {acc_v1:.3f}")
print(f"   {'v1.1 (200/15)':<15} {precision_v2:.3f}       {recall_v2:.3f}       {f1_v2:.3f}       {acc_v2:.3f}")
print("   " + "-"*60)

# Bestäm preliminär vinnare
if f1_v2 > f1_v1:
    print(f"\n   🏆 Preliminär vinnare: Version B (v1.1) med F1-score {f1_v2:.3f}")
    print("   ⚠️ Version B ser bättre ut offline – dags för A/B-test i produktion!")
else:
    print(f"\n   🏆 Preliminär vinnare: Version A (v1.0) med F1-score {f1_v1:.3f}")
    print("   ℹ️ Version A är redan bäst – behåll den i produktion.")

# =============================================
# Steg 4: Simulera A/B-test i produktion (trafikdelning)
# =============================================
print("\n" + "="*70)
print("🚦 Steg 4: SIMULERAR A/B-TEST I PRODUKTION (70/30 trafikdelning)")
print("="*70)

# A/B-test parametrar
TRAFFIC_SPLIT = 70  # 70% till Version A, 30% till Version B
TEST_ITERATIONS = 100  # Antal simuleringar

print(f"\n   📊 Konfiguration:")
print(f"      Version A (v1.0): {TRAFFIC_SPLIT}% av trafiken")
print(f"      Version B (v1.1): {100-TRAFFIC_SPLIT}% av trafiken")
print(f"      Totala simuleringar: {TEST_ITERATIONS}")

# Trafikräknare
traffic_v1 = 0
traffic_v2 = 0
correct_v1 = 0
correct_v2 = 0
predictions_v1 = []
predictions_v2 = []

print("\n   📋 Pågående A/B-test...")
print("   " + "-"*55)

for i in range(TEST_ITERATIONS):
    user_id = f"user_{i}"
    # Hash-baserad routing (samma användare får alltid samma modell)
    hash_val = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100
    
    # Välj en random testrad
    idx = np.random.randint(0, len(X_test_scaled))
    test_scaled = X_test_scaled[idx:idx+1]
    true_label = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]
    
    if hash_val < TRAFFIC_SPLIT:  # Version A
        traffic_v1 += 1
        pred = model_v1.predict(test_scaled)[0]
        predictions_v1.append(pred)
        if pred == true_label:
            correct_v1 += 1
    else:  # Version B
        traffic_v2 += 1
        pred = model_v2.predict(test_scaled)[0]
        predictions_v2.append(pred)
        if pred == true_label:
            correct_v2 += 1
    
    # Skriv ut var 20:e iteration
    if (i + 1) % 20 == 0:
        print(f"      {i+1} användare bearbetade...")

# =============================================
# Steg 5: Analysera A/B-test resultat
# =============================================
print("\n" + "="*70)
print("📊 Steg 5: ANALYS AV A/B-TEST RESULTAT")
print("="*70)

print("\n   📊 Trafikfördelning:")
print(f"      Version A (v1.0): {traffic_v1} användare ({traffic_v1/TEST_ITERATIONS:.0%})")
print(f"      Version B (v1.1): {traffic_v2} användare ({traffic_v2/TEST_ITERATIONS:.0%})")

print("\n   📊 Modellprestanda i A/B-testet:")
if traffic_v1 > 0:
    acc_v1_live = correct_v1 / traffic_v1
    print(f"      Version A (v1.0): {correct_v1}/{traffic_v1} korrekt ({acc_v1_live:.1%})")
if traffic_v2 > 0:
    acc_v2_live = correct_v2 / traffic_v2
    print(f"      Version B (v1.1): {correct_v2}/{traffic_v2} korrekt ({acc_v2_live:.1%})")

# =============================================
# Steg 6: SLUTSATS OCH REKOMMENDATION
# =============================================
print("\n" + "="*70)
print("🏆 SLUTSATS OCH REKOMMENDATION")
print("="*70)

if f1_v2 > f1_v1:
    print("""
   ✅ Version B (v1.1) presterar BÄTTRE offline!
   
   Rekommendation i produktion:
   1. Fortsätt A/B-testet i 1-2 veckor med riktig användardata
   2. Samla tillräckligt med statistik
   3. Om Version B fortsatt är bättre → rulla ut till 100% av trafiken
   4. Uppdatera MLflow med den nya modellen som 'produktion'
   
   🚀 Nästa steg: Träna version v1.2 med n_estimators=300, max_depth=20
""")
else:
    print("""
   ✅ Version A (v1.0) presterar FORTFARANDE BÄST!
   
   Rekommendation i produktion:
   1. Behåll Version A (v1.0) i produktion
   2. Testa andra parametrar istället:
      - Öka max_depth till 20
      - Ändra min_samples_split
      - Justera class_weight
   3. Träna Version C (v1.2) med nya parametrar och A/B-testa igen
   
   🔍 Lärdom: Fler träd (200 vs 100) gav inte bättre resultat i detta fall.
""")

print("\n" + "="*70)
print("✅ A/B-TEST RAMVERKET ÄR KLART!")
print("="*70)
print("""
┌─────────────────────────────────────────────────────────────────┐
│  VAD HAR VI GJORT I DETTA PRODUKTIONS-A/B-TEST?                 │
├─────────────────────────────────────────────────────────────────┤
│  ✅ Jämfört två versioner av SAMMA modelltyp (Random Forest)    │
│     Version A: n_estimators=100, max_depth=10 (din bästa)       │
│     Version B: n_estimators=200, max_depth=15 (ny version)      │
│                                                                 │
│  ✅ Offline-jämförelse på testdata innan A/B-test               │
│  ✅ Simulerat A/B-test med 70/30 trafikdelning                  │
│  ✅ Hash-baserad routing (samma användare → samma modell)       │
│  ✅ Rekommendation baserad på resultat                          │
└─────────────────────────────────────────────────────────────────┘

🔍 I verklig produktion:
   1. Skicka 10-20% av trafiken till nya versionen (canary deployment)
   2. Samla data i 1-2 veckor
   3. Analysera metrics (konvertering, precision, recall)
   4. Rulla ut vinnaren till 100% eller testa nya parametrar
""")

# **`STEG 12: FULL AUTOMATISERING MED SLACK-RAPPORT & AI-ANALYS`**
**Detta är en produktions-klar pipeline som:**
1. Tränar Random Forest-modell
2. Jämför med historisk prestanda (via MLflow)
3. Skickar ALLTID en Slack-rapport med LLM-genererad analys
4. Sparar historik i CSV och MLflow för spårbarhet

**API-nycklar som behövs (frågas i terminalen):**
- Slack Webhook URL (från api.slack.com/apps)
- Groq API Key (från console.groq.com)
- Snowflake lösenord

In [0]:
# =============================================
# STEG 12: FULL AUTOMATISERING MED SLACK-RAPPORT + LLM-ANALYS
# =============================================
# Detta gör:
# 1. Tränar Random Forest-modell
# 2. Jämför med historisk prestanda
# 3. Skickar ALLTID en Slack-rapport med LLM-genererad analys
# 4. Sparar historik i CSV och MLflow
# =============================================

print("="*70)
print("🚀 FULL AUTOMATISERING MED SLACK-RAPPORT + LLM-ANALYS")
print("="*70)

# =============================================
# Steg 0: Fråga efter alla API-nycklar och URL:er
# =============================================
print("\n" + "="*70)
print("📢 KONFIGURATION – Vänligen ange följande:")
print("="*70)

from getpass import getpass
import requests
import json

# Slack Webhook URL
print("\n🔗 SLACK WEBHOOK URL")
print("   Så här får du den: https://api.slack.com/apps → din app → Incoming Webhooks")
slack_webhook_url = input("   Klistra in din Slack Webhook URL: ").strip()

# Groq API-nyckel (för LLM-analys)
print("\n🤖 GROQ API NYCKEL (för AI-analys)")
print("   Så här får du den: console.groq.com → API Keys → Create API Key")
groq_api_key = getpass("   Ange din Groq API-nyckel: ")

# Snowflake lösenord
print("\n❄️ SNOWFLAKE LÖSENORD")
snowflake_password = getpass("   Ange ditt Snowflake-lösenord: ")

print("\n✅ Alla inställningar mottagna! Startar automation...")

# =============================================
# Steg 1: Importera bibliotek
# =============================================
print("\n📚 Importerar bibliotek...")
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
import joblib
from datetime import datetime
import snowflake.connector as sf
from groq import Groq

# =============================================
# Steg 2: Hämta data från Snowflake
# =============================================
print("\n📊 Steg 2: Hämtar data från Snowflake...")

conn = sf.connect(
    account='jnqaoob-to29395',
    user='ARSENIY',
    password=snowflake_password,
    warehouse='COMPUTE_WH',
    database='INSIDER_THREAT_DB',
    schema='DBT_AB'
)

df = pd.read_sql("SELECT * FROM FCT_FEATURES", conn)
conn.close()
print(f"✅ Hämtade {len(df)} rader")

# =============================================
# Steg 3: Förbered data
# =============================================
print("\n🤖 Steg 3: Förbereder data...")
y = (df['ATTACK_TYPE'] != 'normal').astype(int)
X = df.select_dtypes(include=['number'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"✅ {len(X_train)} träningsrader, {len(X_test)} testrader")

# =============================================
# Steg 4: Träna modell
# =============================================
print("\n🌲 Steg 4: Tränar Random Forest...")
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)
print("✅ Modell tränad")

# =============================================
# Steg 5: Beräkna prestanda
# =============================================
print("\n📊 Steg 5: Beräknar prestanda...")
y_pred = model.predict(X_test_scaled)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"   Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"   Recall: {recall:.3f} ({recall*100:.1f}%)")
print(f"   F1-score: {f1:.3f}")

# =============================================
# Steg 6: Hämta historisk prestanda från MLflow
# =============================================
print("\n🔍 Steg 6: Hämtar historisk prestanda...")

client = MlflowClient()
experiment_name = "/Users/arsen1222022@gmail.com/insider_threat_automation"

try:
    mlflow.create_experiment(experiment_name)
except:
    pass

mlflow.set_experiment(experiment_name)

latest_f1 = None
historical_f1_list = []

try:
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment:
        runs = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=["start_time DESC"],
            max_results=10
        )
        
        for _, row in runs.iterrows():
            if 'metrics.f1_score' in row and pd.notna(row['metrics.f1_score']):
                historical_f1_list.append(row['metrics.f1_score'])
        
        if historical_f1_list:
            latest_f1 = historical_f1_list[0]
            historical_avg_f1 = np.mean(historical_f1_list)
            historical_max_f1 = np.max(historical_f1_list)
        else:
            latest_f1 = None
except Exception as e:
    print(f"   ⚠️ Kunde inte hämta historik: {e}")

# =============================================
# Steg 7: Beräkna förändring och status
# =============================================
if latest_f1 is not None:
    change_percent = ((f1 - latest_f1) / latest_f1) * 100
    print(f"\n   📊 Jämförelse med senaste modellen:")
    print(f"      Senaste F1-score: {latest_f1:.3f}")
    print(f"      Ny F1-score: {f1:.3f}")
    print(f"      Förändring: {change_percent:+.2f}%")
    
    if change_percent > 1:
        status = "✅ FÖRBÄTTRING"
        status_emoji = "📈"
    elif change_percent < -1:
        status = "⚠️ FÖRSÄMRING"
        status_emoji = "📉"
    else:
        status = "✅ STABIL"
        status_emoji = "⚖️"
else:
    change_percent = 0
    status = "✅ FÖRSTA KÖRNINGEN"
    status_emoji = "🆕"

# =============================================
# Steg 8: Generera LLM-analys med Groq
# =============================================
print("\n🤖 Steg 8: Genererar AI-analys med Groq...")

groq_client = Groq(api_key=groq_api_key)

analysis_prompt = f"""
Du är en expert på maskininlärning och cybersäkerhet. Analysera denna Random Forest-modell för insiderhot-detektering:

MODELLPRESTANDA:
- Precision: {precision:.3f} ({precision*100:.1f}%) – Andel flaggade som verkligen var attacker
- Recall: {recall:.3f} ({recall*100:.1f}%) – Andel attacker som upptäcktes
- F1-score: {f1:.3f} – Balans mellan precision och recall

HISTORISK JÄMFÖRELSE:
- Senaste F1-score: {latest_f1 if latest_f1 else 'Ingen tidigare'}
- Förändring: {change_percent:+.2f}%
- Status: {status}

Tidigare F1-score värden: {historical_f1_list[:3] if historical_f1_list else 'Ingen historik'}

Skriv en KORT professionell analys (max 5 meningar) på svenska. Inkludera:
1. En övergripande bedömning av modellens kvalitet
2. En kommentar om förändringen jämfört med tidigare
3. En konkret rekommendation

Använd ett proffsigt men lättförståeligt språk.
"""

try:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": analysis_prompt}],
        temperature=0.5
    )
    llm_analysis = response.choices[0].message.content
    print("   ✅ AI-analys genererad!")
except Exception as e:
    print(f"   ⚠️ Kunde inte generera AI-analys: {e}")
    llm_analysis = f"Modellen uppnår {precision*100:.1f}% precision och {recall*100:.1f}% recall. F1-score: {f1:.3f}. {'Förbättring' if change_percent > 0 else 'Stabil' if change_percent == 0 else 'Försämring'} jämfört med tidigare modell."

# =============================================
# Steg 9: Skapa Slack-meddelande
# =============================================
print("\n📧 Steg 9: Skickar Slack-rapport...")

slack_message = f"""{status_emoji} *INSIDER THREAT MODELL – AUTOMATISERAD RAPPORT* {status_emoji}

*📊 MODELLPRESTANDA:*
├── Precision: {precision*100:.1f}% (av flaggade var attacker)
├── Recall: {recall*100:.1f}% (av attacker upptäcktes)
├── F1-score: {f1:.3f}
└── Status: {status}

*📈 HISTORISK JÄMFÖRELSE:*
├── Senaste F1-score: {latest_f1 if latest_f1 else 'Ingen tidigare'}
├── Förändring: {change_percent:+.2f}%
└── Antal historiska körningar: {len(historical_f1_list)}

*🤖 AI-ANALYS (Groq Llama 3.3):*
{llm_analysis}

*📅 Tid:* {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
*🏷️ Job:* Insider Threat Automation
"""

try:
    response = requests.post(slack_webhook_url, json={"text": slack_message})
    if response.status_code == 200:
        print("   ✅ Slack-rapport skickad!")
        print("\n   🔍 Hitta rapporten i Slack:")
        print("   ┌─────────────────────────────────────────────────────────────┐")
        print("   │ 1. Öppna Slack-appen eller webbversionen                    │")
        print("   │ 2. Gå till kanalen: #all-insider-threat-detection          │")
        print("   │ 3. Leta efter meddelandet från 'Insider Threat Alerts'     │")
        print("   └─────────────────────────────────────────────────────────────┘")
    else:
        print(f"   ❌ Slack-fel: {response.status_code}")
except Exception as e:
    print(f"   ❌ Kunde inte skicka: {e}")

# =============================================
# Steg 10: Logga till MLflow
# =============================================
print("\n📦 Steg 10: Loggar till MLflow...")

run_name = f"Automated_Job_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(run_name=run_name):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(model, "random_forest_model")
    joblib.dump(scaler, "scaler.pkl")
    mlflow.log_artifact("scaler.pkl")
    mlflow.set_tag("job_type", "automated_monitoring_with_slack")

print(f"✅ Loggat som: {run_name}")

# =============================================
# Steg 11: Spara till CSV
# =============================================
print("\n💾 Steg 11: Sparar historik...")

history_file = "model_performance_history.csv"
performance_record = pd.DataFrame([{
    'timestamp': datetime.now(),
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'n_estimators': 100,
    'max_depth': 10,
    'change_percent': change_percent,
    'status': status
}])

try:
    existing = pd.read_csv(history_file)
    updated = pd.concat([existing, performance_record], ignore_index=True)
    updated.to_csv(history_file, index=False)
    print(f"✅ Historik uppdaterad ({len(updated)} körningar totalt)")
except:
    performance_record.to_csv(history_file, index=False)
    print("✅ Ny historikfil skapad")

# =============================================
# Steg 12: Slutrapport
# =============================================
print("\n" + "="*70)
print("📊 SLUTRAPPORT – FULL AUTOMATISERING")
print("="*70)
print(f"""
┌─────────────────────────────────────────────────────────────────┐
│  AUTOMATISERINGSJOB – SAMMANFATTNING                            │
├─────────────────────────────────────────────────────────────────┤
│  Datum/Tid: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
│  Modell: Random Forest (n_estimators=100, max_depth=10)
│                                                                 │
│  PRESTANDA:                                                     │
│  ├── Precision: {precision*100:.1f}%
│  ├── Recall: {recall*100:.1f}%
│  └── F1-score: {f1:.3f}
│                                                                 │
│  HISTORIK:                                                      │
│  ├── Status: {status}
│  ├── Förändring: {change_percent:+.2f}%
│  └── Totalt körningar: {len(historical_f1_list) + 1}
│                                                                 │
│  SLACK: ✅ Rapport skickad
│  MLflow: ✅ Modell sparad
│  CSV: ✅ Historik sparad
└─────────────────────────────────────────────────────────────────┘
""")

print("\n" + "="*70)
print("✅ FULL AUTOMATISERING KLAR!")
print("="*70)

# **`STEG 13: SKAPA PERMANENT SQL-TABELL OCH DASHBOARD`**
**Detta gör:**
1. Skapar en permanent SQL-tabell från historik-CSV
2. Tabellen syns i SQL Warehouse för dashboards
3. Skriver ut instruktioner för att skapa dashboard


Efter att koden körts, följ instruktionerna i terminalen
för att skapa dashboard i Databricks SQL!

In [0]:
# =============================================
# STEG 13.1: SKAPA PERMANENT SQL-TABELL FRÅN HISTORIK-CSV
# =============================================
# Detta skapar en tabell som syns i SQL Warehouse för dashboards
# =============================================

print("="*60)
print("📊 Steg 13.1: Skapar permanent SQL-tabell från historik")
print("="*60)

import pandas as pd

# =============================================
# Steg 1: Läs in historik-CSV
# =============================================
print("\n📂 Steg 1: Läser in historik-CSV...")

try:
    df_history = pd.read_csv("model_performance_history.csv")
    print(f"✅ Hittade {len(df_history)} historiska körningar")
    print(f"   Kolumner: {list(df_history.columns)}")
except FileNotFoundError:
    print("❌ Ingen historikfil hittades – kör Steg 12 först!")
    raise

# =============================================
# Steg 2: Skapa Spark DataFrame
# =============================================
print("\n🔥 Steg 2: Skapar Spark DataFrame...")
spark_df = spark.createDataFrame(df_history)
print("✅ Spark DataFrame skapad")

# =============================================
# Steg 3: Skapa PERMANENT tabell för SQL Warehouse
# =============================================
print("\n💾 Steg 3: Skapar permanent tabell för SQL Warehouse...")

# Spara som permanent tabell (syns i SQL Warehouse)
spark_df.write.mode("overwrite").saveAsTable("model_performance_history")

print("✅ Permanent tabell 'model_performance_history' skapad!")
print("   📍 Tabellen syns nu i:")
print("      - SQL Warehouse → Databricks SQL")
print("      - Databricks → Catalog → default → model_performance_history")

# =============================================
# Steg 4: Skapa även temp-view för notebook-användning
# =============================================
print("\n📝 Steg 4: Skapar temp-view för notebook...")
spark_df.createOrReplaceTempView("model_performance_history_temp")
print("✅ Temp-view 'model_performance_history_temp' skapad för notebook")

# =============================================
# Steg 5: Visa de senaste körningarna
# =============================================
print("\n📋 Steg 5: Senaste körningarna:")
print("-" * 50)
display(spark_df.tail(5))

# =============================================
# Steg 6: Verifiera att tabellen fungerar
# =============================================
print("\n🔍 Steg 6: Verifierar tabellen...")

# Testa att läsa från den permanenta tabellen
test_df = spark.sql("SELECT COUNT(*) as count FROM model_performance_history")
test_count = test_df.collect()[0]['count']
print(f"✅ Permanent tabell innehåller {test_count} rader")

print("\n" + "="*60)
print("✅ KLART! Du kan nu använda tabellen i SQL Warehouse!")
print("="*60)

# =============================================
# SKRIV UT INSTRUKTIONER FÖR DASHBOARD I TERMINALEN
# =============================================
print("\n" + "="*60)
print("📌 INSTRUKTIONER FÖR ATT SKAPA DASHBOARD")
print("="*60)

print("""
🔹 STEG 1: Skapa dataset i SQL Warehouse

   1. Gå till vänstermenyn → "SQL" → "SQL Editor"
   2. Välj din SQL Warehouse (t.ex. "Serverless Starter Warehouse")
   3. Klicka på "New Query"
   4. Klistra in SQL-frågan:

   SELECT 
       timestamp,
       ROUND(precision * 100, 1) as precision_percent,
       ROUND(recall * 100, 1) as recall_percent,
       ROUND(f1_score, 3) as f1_score,
       COALESCE(status, 'Ingen status') as status
   FROM model_performance_history
   ORDER BY timestamp DESC;

   5. Klicka "Run" → "Save" (namn: "Model Performance Data")

🔹 STEG 2: Skapa dashboard

   1. Gå till vänstermenyn → "SQL" → "Dashboards"
   2. Klicka "Create Dashboard" (namn: "Insider Threat Monitoring")
   3. Klicka "Add" → "Visualization" → Välj datasetet

🔹 STEG 3: Skapa linjediagram (prestanda över tid)

   1. Välj "Line Chart"
   2. X-axel: timestamp
   3. Y-axel: precision_percent, recall_percent, f1_score
   4. Klicka "Save"

🔹 STEG 4: Skapa kortvy (senaste värden)

   1. Klicka "Add" → "Visualization"
   2. Välj "Counter"
   3. Välj f1_score som värde
   4. Klicka "Save"

🔹 STEG 5: Skapa tabell (alla körningar)

   1. Klicka "Add" → "Visualization"
   2. Välj "Table"
   3. Välj kolumner: timestamp, precision_percent, recall_percent, f1_score, status
   4. Klicka "Save"

🔹 ELLER ANVÄND AI-ASSISTENTEN (ENKLARE!)

   1. Gå till din dashboard
   2. Klicka på den blå AI-knappen (Chat-bot)
   3. Skriv: "Create a visualization of: Model Performance Metrics Over Time Summary"
   4. AI skapar automatiskt alla visualiseringar åt dig!
""")

print("\n" + "="*60)
print("✅ DASHBOARD-INSTRUKTIONER KLARA!")
print("="*60)